# XGBoost regression pack — diabetes progression

**Kya chahiye:** Google Colab ya local Python 3.10+. `Run all` karo; first cell missing packages install karta hai.

**Dataset:** scikit-learn ka canonical `load_diabetes` (442 rows, 10 baseline features, continuous one-year progression target). Koi external data download nahi.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "sklearn": "scikit-learn==1.5.2",
    "xgboost": "xgboost==2.1.3",
    "pandas": "pandas==2.2.3",
}
missing = [pip_name for module, pip_name in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import sklearn
import xgboost
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("Runtime is ready. This pack is for Colab or a local environment, not the offline Code Bro kernel.")

## 1. Load and split

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

data = load_diabetes(as_frame=True)
df = data.frame
X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("shape:", df.shape)
print("train:", X_train.shape, "| test:", X_test.shape)
print("target mean:", round(float(y_train.mean()), 2), "| std:", round(float(y_train.std()), 2))

## 2. Baselines and one metric function

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)
    print(f"{name:>18}: MAE={mae:.2f} | RMSE={rmse:.2f} | R²={r2:.3f}")
    return pred

linear = LinearRegression()
linear_pred = evaluate("linear baseline", linear)

forest = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)
forest_pred = evaluate("random forest", forest)

xgb = XGBRegressor(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=4,
)
xgb_pred = evaluate("xgboost", xgb)

## 3. Inspect the residuals, not only the score

In [ ]:
residuals = y_test.to_numpy() - xgb_pred
worst = np.argsort(np.abs(residuals))[::-1][:5]
for row in worst:
    print(f"actual={y_test.to_numpy()[row]:7.1f} | predicted={xgb_pred[row]:7.1f} | residual={residuals[row]:7.1f}")
print("mean residual:", round(float(residuals.mean()), 2))
print("residual std :", round(float(residuals.std()), 2))

## 4. Feature importance is a diagnostic, not a causal claim

In [ ]:
import pandas as pd

importance = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.round(3).to_string())

## 5. Honest limitations

- The target is disease progression, not a causal effect of any feature. Coefficients and importances are not treatments.
- This split is random. A real clinical model should be validated on a separate cohort and a later time period.
- XGBoost's default feature importance is based on usage/splits; correlated features can split credit.
- Hyperparameters here are a small, documented starting point, not a tuned production configuration.
- Keep the random seed and package versions with the result if you want the run to be reproducible.

**Exercise:** add early stopping with a validation split and compare the best iteration with the fixed 500-tree model. Report MAE and the number of trees actually used.